Import das Bibliotecas Necessarias para etl como Pandas , ftplib , os , py7zr e criação da função conectar no servidor ftp e navegar nas pastas ate a pasta Novo caged e no ano 2026 

In [2]:
from ftplib import FTP
import pandas as pd
import os
import py7zr

In [22]:
import pandas as pd

caminho = r'C:\Users\gabriel.amaral\Desktop\ETL_EMPREGO\Layout_Novo_Caged_Movimentacao.xlsx'

excel = pd.ExcelFile(caminho)

lista_abas = excel.sheet_names

print(lista_abas)

['Layout', 'região', 'uf', 'município', 'seção', 'subclasse', 'categoria', 'cbo2002ocupação', 'graudeinstrução', 'raçacor', 'sexo', 'tipoempregador', 'tipoestabelecimento', 'tipomovimentação', 'tipodedeficiência', 'indtrabintermitente', 'indtrabparcial', 'tamestabjan', 'indicadoraprendiz', 'origemdainformação', 'indicadordeexclusão', 'indicadordeforadoprazo', 'unidadesaláriocódigo']


In [ ]:
df = pd.read_excel(caminho,sheet_name='município')
df.to_csv('municipio.csv',index=False,encoding='utf-8-sig')

In [ ]:
def conectar(ano = None , competencia = None):
    ftp = FTP('ftp.mtps.gov.br')
    ftp.encoding = 'utf-8'
    ftp.login()
    if ano and competencia:
        ftp.cwd(f'pdet/microdados/NOVO CAGED/{ano}/{competencia}')
    else:
        ftp.cwd('pdet/microdados/NOVO CAGED')
    return ftp


def listar_anos():
    ftp = conectar()
    anos = ftp.nlst()
    ftp.quit()
    return sorted(anos)

def listar_meses(ano):
    ftp = conectar()
    ftp.cwd(ano)
    meses = ftp.nlst()
    ftp.quit()
    return sorted(meses)

def escolher_competencia():
    anos = listar_anos()
    print('Anos disponíveis:', anos)
    ano_escolhido = input('Digite o ano que deseja acessar: ').strip()
    
    meses = listar_meses(ano_escolhido)
    print(f'Meses disponíveis em {ano_escolhido}: ', meses)
    escolha = input('Digite os meses separados por vírgula (ex: 202601 , 202602), ou  "todos" para baixar tudo: ').strip()
    
    if escolha.lower() == 'todos':
        return ano_escolhido , meses
    else:
        selecionados = [m.strip() for m in escolha.split(',')]
        invalidos = [m for m in selecionados if m not in meses]
        if invalidos:
            print(f'Aviso: esses meses não existem na pasta e serão ignorados: {invalidos}')
        return ano_escolhido , [m for m in selecionados if m in meses]
    
    

In [ ]:
def processar_competencia(ano,competencia):
    """Ano: ex '2026'
    Competencia: ex '202601' """
    pas_destino = r'C:\Users\gabriel.amaral\Desktop\ETL_EMPREGO\Arquivos'
    os.makedirs(pas_destino,exist_ok= True)
    arquivos  = [
        f'CAGEDEXC{competencia}.7z',
        f'CAGEDFOR{competencia}.7z',
        f'CAGEDMOV{competencia}.7z'
    ]
    
    ftp = FTP()

In [36]:
pas_destino = r'C:\Users\gabriel.amaral\Desktop\ETL_EMPREGO\Arquivos'
os.makedirs(pas_destino, exist_ok=True)

arquivos = ['CAGEDEXC202601.7z', 'CAGEDFOR202601.7z', 'CAGEDMOV202601.7z']
ftp = conectar()

for nome_arquivo in arquivos:
    caminho_local = os.path.join(pas_destino, nome_arquivo)  # mantém .7z
    with open(caminho_local, 'wb') as f:
        ftp.retrbinary(f'RETR {nome_arquivo}', f.write)
    print(f'Baixado: {nome_arquivo}')

ftp.quit()


for nome_7z in arquivos:
    caminho_7z = os.path.join(pas_destino, nome_7z)
    with py7zr.SevenZipFile(caminho_7z, mode='r') as z:
        print(nome_7z, '->', z.getnames())
        z.extractall(path=pas_destino)

Baixado: CAGEDEXC202601.7z
Baixado: CAGEDFOR202601.7z
Baixado: CAGEDMOV202601.7z
CAGEDEXC202601.7z -> ['CAGEDEXC202601.txt']
CAGEDFOR202601.7z -> ['CAGEDFOR202601.txt']
CAGEDMOV202601.7z -> ['CAGEDMOV202601.txt']


In [40]:
arquivos_txt = ['CAGEDEXC202601.txt', 'CAGEDFOR202601.txt', 'CAGEDMOV202601.txt']
for nome_txt in arquivos_txt:
    caminho_txt = os.path.join(pas_destino,nome_txt)
    df = pd.read_csv(
        caminho_txt,
        sep=';',
        encoding='utf-8',
        decimal = ',',
        low_memory=False
    )
    caminho_csv = caminho_txt.replace('.txt' , '.csv')
    df.to_csv(caminho_csv,index=False, sep=';', encoding='utf-8-sig')

In [43]:
pas_destino = r'C:\Users\gabriel.amaral\Desktop\ETL_EMPREGO\Arquivos'
for nome_arquivo in os.listdir(pas_destino):
    if not nome_arquivo.endswith('.csv'):
        caminho = os.path.join(pas_destino,nome_arquivo)
        os.remove(caminho)
        print(f'Arquivo Removido : {nome_arquivo}')

Arquivo Removido : CAGEDEXC202601.7z
Arquivo Removido : CAGEDEXC202601.txt
Arquivo Removido : CAGEDFOR202601.7z
Arquivo Removido : CAGEDFOR202601.txt
Arquivo Removido : CAGEDMOV202601.7z
Arquivo Removido : CAGEDMOV202601.txt
